In [11]:
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
import time
import random
from bs4 import BeautifulSoup as bs
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

import pandas as pd
import random
from tqdm import tqdm
import logging

### 키워드 설정

In [12]:
keywords = ["진주성", "남강", "엠비씨네", "롯데시네마", "CGV", "메가박스", "이마트", "홈플러스", "롯데몰", "롯데마트", "탑마트", "갤러리아", "탑마트", "LH", "모다아울렛", "경상국립대", "경상대", "중앙시장", "진주역", "칠암캠", "가좌캠"]

### 로그인 설정

In [13]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager

driver = webdriver.Chrome()

# 네이버 로그인 화면 이동
login_url = 'https://nid.naver.com/nidlogin.login'
driver.get(login_url)
driver.implicitly_wait(5)

# 아이디& 비밀번호 입력
my_id = "sc1645"  # "id" 대신에 자신의 네이버 아이디 입력
my_pw = "chanS1501!"  # "password" 대신에 자신의 네이버 비밀번호 입력

# 로그인 id, pw 입력
# 네이버에 로그인 할 경우 'send_keys()' 함수가 아니라 'execute_script()' 함수를 사용
driver.execute_script("document.getElementsByName('id')[0].value = \'" + my_id + "\'")
driver.execute_script("document.getElementsByName('pw')[0].value = \'" + my_pw + "\'")
time.sleep(1)

# '로그인' 버튼 클릭
driver.find_element(By.XPATH, '//*[@id="log.login"]').click()
time.sleep(random.uniform(1,1.7))
url = 'https://cafe.naver.com/lgtabbook' # 크롤링할 카페 url 입력
driver.get(url)
time.sleep(1)

In [14]:
# 날짜 수집 함수
def extract_date(soup):
    try:
        date_element = soup.select_one('span.date')
        if date_element:
            date_text = date_element.text.strip()
            # 날짜 형식 처리
            if "시간 전" in date_text or "분 전" in date_text or "방금" in date_text:
                # 상대적 시간은 현재 날짜로 처리
                return time.strftime('%Y-%m-%d', time.localtime())
            elif "." in date_text:
                # YYYY.MM.DD 형식 -> YYYY-MM-DD로 변환
                return date_text.replace(".", "-")
        return "날짜 없음"  # 기본값
    except Exception as e:
        logging.error(f"날짜 추출 오류: {e}")
        return "날짜 없음"

# 본문 수집 함수
def extract_content(soup, include_images=True):
    contents = []

    # 다양한 본문 선택자 처리
    selectors = [
        'div.se-module.se-module-text',  # 일반 텍스트 본문
        'div.ContentRenderer',           # 특수 서식 본문
        'div.scrap_added',               # 스크랩 본문
    ]
    for selector in selectors:
        elements = soup.select(selector)
        if elements:
            contents.extend([element.text.strip() for element in elements])

    # 이미지 포함 여부에 따른 처리
    if include_images:
        images = soup.select('img')
        for img in images:
            if img.get('src'):
                contents.append(f"[이미지: {img['src']}]")  # 이미지 URL 포함

    return ' '.join(contents).strip() if contents else "본문 없음"

In [15]:
import pandas as pd

# 제목, 본문, 댓글, 날짜, 검색 키워드, 게시판 이름 - 빈 list 생성
titles = []
reviews = []
comments = []
dates = []
search_keywords = []
board_titles = []
urls = []  # 게시글 URL 저장

time.sleep(random.uniform(1, 1.7))
url = 'https://cafe.naver.com/lgtabbook'  # 크롤링할 카페 url 입력
driver.get(url)
time.sleep(1)

time.sleep(1)
for k in tqdm(keywords):
    print(k, "키워드 크롤링 중입니다")

    # 검색
    search_box = driver.find_element(By.XPATH, '//*[@id="topLayerQueryInput"]')
    search_box.send_keys(k)
    search_box.send_keys(Keys.RETURN)
    time.sleep(1)

    driver.switch_to.frame('cafe_main')

    # 데이터 수집기간 설정
    time_box = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="currentSearchDateTop"]')))
    driver.execute_script("arguments[0].click();", time_box)

    # 시작 날짜 입력
    start_time = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="input_1_top"]')))
    start_time.click()
    time.sleep(1)
    start_time.send_keys('20200101')

    # 종료 날짜 입력
    end_time = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="input_2_top"]')))
    end_time.click()
    time.sleep(1.5)
    end_time.send_keys('20241130')

    # 설정 버튼 클릭
    btn_set_top = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="btn_set_top"]')))
    driver.execute_script("arguments[0].click();", btn_set_top)
    time.sleep(1.5)

    # 검색 버튼 클릭
    search_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, '//*[@id="main-area"]/div[1]/div[1]/form/div[4]/button')))
    driver.execute_script("arguments[0].click();", search_button)
    time.sleep(1.5)

    try:
        if driver.find_element(By.XPATH, '//*[@id="main-area"]/div[5]/table/tbody/tr/td/div').text.strip() == '등록된 게시글이 없습니다.':
            driver.get(url)
            time.sleep(1)
            continue
    except:
        print("크롤링 시작합미당~")
        pass

    # 50개씩 보기
    driver.find_element(By.XPATH, '//*[@id="listSizeSelectDiv"]').click()
    driver.find_element(By.XPATH, '//*[@id="listSizeSelectDiv"]/ul/li[7]/a').click()

    # url 가져오기
    page_url_list = [a.get_attribute('href') for a in driver.find_elements(By.CSS_SELECTOR, 'a.article')]
    first_page = driver.find_element(By.CSS_SELECTOR, 'a.on').get_attribute('href')

    # 페이지 넘기기 검색시 최대 100페이지
    for j in range(1, 11):
        try:
            if j % 8 == 0:
                # 다음 링크 가져오기
                time.sleep(1)
                link = first_page[:-1] + str(j)

                driver.quit()
                driver = webdriver.Chrome()  # 현재 컴퓨터 크롬드라이버 위치로 변경

                # 재로그인
                login_url = 'https://nid.naver.com/nidlogin.login?mode=form&url=https%3A%2F%2Fwww.naver.com'
                driver.get(login_url)
                driver.implicitly_wait(10)

                driver.execute_script("document.getElementsByName('id')[0].value = \'" + my_id + "\'")
                driver.execute_script("document.getElementsByName('pw')[0].value = \'" + my_pw + "\'")
                time.sleep(1)

                # '로그인' 버튼 클릭
                driver.find_element('id', 'log.login').click()
                time.sleep(1)

                # 다음 링크부터 가져오기
                driver.get(link)
                driver.switch_to.frame('cafe_main')

        except:
            pass

        try:
            if driver.find_element(By.XPATH, '//*[@id="main-area"]/div[5]/table/tbody/tr/td/div').text.strip() == '등록된 게시글이 없습니다.':
                break
        except:
            pass

        if j > 1:
            next_page = first_page[:-1] + str(j)
            driver.get(next_page)
            time.sleep(1)
            driver.switch_to.frame('cafe_main')

        page_url_list = [a.get_attribute('href') for a in driver.find_elements(By.CSS_SELECTOR, 'a.article')]

        for url in page_url_list:
            try:
                driver.get(url)
                time.sleep(random.uniform(1, 1.7))
                driver.switch_to.frame('cafe_main')
            except:
                continue

            try:
                # 게시판 이름 수집
                board_title = driver.find_element(By.CSS_SELECTOR, 'div.ArticleTitle').text
                board_titles.append(board_title)

                # 제목 수집
                title = driver.find_element(By.CSS_SELECTOR, 'h3.title_text').text
                titles.append(title)

                # 날짜 수집 (함수 적용)
                soup = bs(driver.page_source, 'lxml')
                day = extract_date(soup)
                dates.append(day)

                # 본문 수집 (함수 적용)
                content = extract_content(soup)
                reviews.append(content)

                # 키워드 수집
                search_keywords.append(k)

                # URL 수집
                urls.append(url)

            except:
                continue

            # 댓글 수집
            try:
                iscomment = soup.find_all('span', class_='text_comment')
                if len(iscomment) == 0:
                    comment = '댓글 없음'
                else:
                    comment = '" '.join([c.text.strip() for c in iscomment])  # 댓글을 큰따옴표로 구분
                comments.append(comment)

            except Exception as e:
                logging.error(f"댓글 수집 중 오류 발생: {e}")
                continue

    driver.get(url)
    time.sleep(1)

  0%|          | 0/21 [00:00<?, ?it/s]

진주성 키워드 크롤링 중입니다


  5%|▍         | 1/21 [05:59<1:59:55, 359.78s/it]

남강 키워드 크롤링 중입니다


 10%|▉         | 2/21 [17:39<2:57:15, 559.77s/it]

...

 90%|█████████ | 19/21 [2:18:54<12:58, 389.39s/it]

칠암캠 키워드 크롤링 중입니다


 95%|█████████▌| 20/21 [2:19:35<04:44, 284.91s/it]

가좌캠 키워드 크롤링 중입니다


100%|██████████| 21/21 [2:19:43<00:00, 399.22s/it]


In [16]:
# DataFrame 생성
data = {
    'date': dates,
    'keyword': search_keywords,
    'title': titles,
    'contents': reviews,
    'comments': comments,
    'board_titles': board_titles,
    'url': urls,
}

df = pd.DataFrame(data)

### url 기준으로 중복 제거 및 저장

In [17]:
#URL 기준 필터링
print(df.shape)
df_drop = df.drop_duplicates(subset='url',keep='first').reset_index(drop=True)
print(df_drop.shape)

(2050, 7)
(2050, 7)


### csv 저장 시 이름 꼭 변경해주세요!!

In [18]:
df_drop.to_csv("./data/review.csv", index=False, encoding = 'utf-8-sig')